# Sandbox del pipeline sin malla -- 100% auto-contenido

Corre TODOS los pasos del pipeline sin malla **excepto el renderizado**
(los renders ya existen en `data/renders/`, se copian, no se regeneran) sobre
un subconjunto chico de 9 objetos (3 BUENO / 3 MEDIO / 3 MALO por
`angular_error` con `axis_v06_nomesh`):

```
1. Copiar .obj/.txt (GT) y renders existentes -> carpeta propia en Experiments/
2. Prompts (texto editable en una celda, no en archivos .txt aparte)
3. Geometria: camara, triangulacion, filtro fondo/objeto (todo inline)
4. Molmo2: carga del modelo + inferencia (inline)          [necesita GPU]
5. Correr inferencia sobre los 9 objetos
6. Estimar eje sin malla (triangulacion, inline)
7. Evaluar contra GT (inline)
8. Resultados por objeto, con su categoria BUENO/MEDIO/MALO
9. Visualizacion 2D/3D (opcional)
```

**Ningun paso llama a un script .py del repo** -- todo el codigo (prompts,
geometria de camara/triangulacion, parseo de la salida de Molmo2, metricas)
esta copiado inline en las celdas de abajo, para poder editarlo directo ahi
y probar variaciones (de prompt, de post-procesamiento, de metrica) sin salir
del notebook. Es una copia funcional de la logica de
`MolmoPointing/molmo_multiview_runner.py`, `pipeline_common/camera.py`,
`pipeline_common/triangulation.py`, `Mapping/estimate_symmetry_no_mesh.py` y
`Mapping/evaluate.py` -- si mas adelante queres llevar una variante ganadora
al pipeline real, hay que trasplantarla a mano a esos archivos.

**No incluye `SDE_ref`/`F1_ref`** (necesitan `gpytoolbox` + muestreo de
superficie -- mas pesado) -- las metricas que si trae (`angular_error`,
`translation_error`, `precision@theta`) alcanzan para comparar variantes
rapido; agregalas a mano si las necesitas.

Los `.obj`/`.txt` y los renders (PNG + `manifest.json` + `metadata_all.json`)
se **copian** desde `data/` (solo lectura, nunca se modifican) -- todo lo
demas (predicciones, ejes estimados, metricas) se genera de cero dentro del
sandbox cada vez que corres las celdas.

## 0. Setup

In [ ]:
import json
import re
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import trimesh
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor

REPO_ROOT = Path(r"C:\Users\HP\Desktop\Seminario de Tesis I\Symmetry-Detection-Using-Multimodal-Vision-Language-Models")
DATA_ROOT = Path(r"C:\Users\HP\Desktop\Seminario de Tesis I\data")  # solo lectura -- fuente de .obj/.txt/renders

SANDBOX_ROOT    = REPO_ROOT / "Experiments" / "sandbox_pipeline_sin_malla_data"
SANDBOX_OBJECTS = SANDBOX_ROOT / "objects"
SANDBOX_RENDERS = SANDBOX_ROOT / "renders"

SYMMETRY_TYPE  = "axis_sym"   # los 9 objetos de abajo son todos axis_sym (verificado)
OBJECTS_SUBDIR = "curated_axis_sym_obj"
SIZE, LIGHTING = 224, "flat"
DEFAULT_FOV    = 60.0

MODEL_ID = "allenai/Molmo2-8B"

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110})

## 1. Lista de objetos + copiar datos al sandbox

Formato de entrada `{CATEGORIA}_{object_id}_2d` -> se parsea a `(categoria, object_id)`.

In [ ]:
RAW_OBJECT_LIST = [
    "BUENO_89cb9b2ad175b833cadf6344ec272e8_2d",
    "BUENO_941271c5d9b192eaccd8f9b9403fd602_2d",
    "BUENO_e0725fd7859fa238ff67c12005f72d2_2d",
    "MALO_6267ef99cbfaea7741cf86c757faf4f9_2d",
    "MALO_955143d7f0b5c70fef76898f881b76a_2d",
    "MALO_e656d6586d481f41eb69804478f9c547_2d",
    "MEDIO_6b8b2cb01c376064c8724d5673a063a6_2d",
    "MEDIO_33b77c66e1f849b790c4e2a44fddf755_2d",
    "MEDIO_f0611ec9ff89209bf10c4513652c1c5e_2d",
]


def parse_entry(entry: str) -> tuple[str, str]:
    """'BUENO_89cb9b2ad175b833cadf6344ec272e8_2d' -> ('BUENO', '89cb9b2ad175b833cadf6344ec272e8').
    El object_id de ShapeNet es siempre hexadecimal de 32 caracteres -- se usa
    eso para separarlo de forma robusta del prefijo de categoria y el sufijo '_2d'."""
    parts = entry.split("_")
    categoria = parts[0]
    object_id = next(p for p in parts[1:] if len(p) >= 30 and all(c in "0123456789abcdef" for c in p))
    return categoria, object_id


OBJECT_LIST = [parse_entry(e) for e in RAW_OBJECT_LIST]  # [(categoria, object_id), ...]
CATEGORY_BY_ID = {oid: cat for cat, oid in OBJECT_LIST}

print(f"{len(OBJECT_LIST)} objetos:")
for cat, oid in OBJECT_LIST:
    print(f"  [{cat:6s}] {oid}")

In [ ]:
SANDBOX_OBJECTS_DIR = SANDBOX_OBJECTS / OBJECTS_SUBDIR
SANDBOX_OBJECTS_DIR.mkdir(parents=True, exist_ok=True)

for cat, oid in OBJECT_LIST:
    # --- .obj / .txt (GT, para el paso de evaluacion -- nunca se usa en la inferencia) ---
    src_objects_dir = DATA_ROOT / "objects" / OBJECTS_SUBDIR
    for ext in (".obj", ".txt"):
        src = src_objects_dir / f"{oid}{ext}"
        dst = SANDBOX_OBJECTS_DIR / f"{oid}{ext}"
        if not dst.exists():
            assert src.exists(), f"No encontrado: {src}"
            shutil.copy2(src, dst)

    # --- renders (PNG + manifest.json + metadata_all.json -- SIN volver a renderizar) ---
    src_render_dir = DATA_ROOT / "renders" / SYMMETRY_TYPE / oid / str(SIZE) / LIGHTING
    dst_render_dir = SANDBOX_RENDERS / SYMMETRY_TYPE / oid / str(SIZE) / LIGHTING
    dst_render_dir.mkdir(parents=True, exist_ok=True)

    assert src_render_dir.exists(), f"No encontrado: {src_render_dir} (¿objeto sin renders generados?)"
    for meta_file in ("manifest.json", "metadata_all.json"):
        src_meta = src_render_dir / meta_file
        dst_meta = dst_render_dir / meta_file
        if src_meta.exists() and not dst_meta.exists():
            shutil.copy2(src_meta, dst_meta)

    n_copied = 0
    for png in src_render_dir.glob("*.png"):
        dst_png = dst_render_dir / png.name
        if not dst_png.exists():
            shutil.copy2(png, dst_png)
            n_copied += 1

    print(f"[{cat:6s}] {oid}: {n_copied} PNG copiados a {dst_render_dir}")

print(f"\nSandbox listo en: {SANDBOX_ROOT}")

## 2. Prompts (editables aca, sin tocar archivos .txt)

Copia funcional de `MolmoPointing/prompts/axis/v06_*.txt` -- edita el texto
directo en esta celda para probar variaciones de redaccion.

In [ ]:
PROMPT_ID = "axis_v06"   # solo para etiquetar el experimento -- cambialo si editas el texto

PROMPT_SINGLE = """You are given ONE image of a 3D object.

The object has ONE dominant rotational symmetry axis.

Your task is to identify the two poles of the rotation axis: the topmost and bottommost points where the axis exits the object's surface.

Return:
- obj_id 1: the TOP pole -- topmost visible point on the rotation axis (at the horizontal center of the topmost surface)
- obj_id 2: the BOTTOM pole -- bottommost visible point on the rotation axis (at the horizontal center of the bottommost surface)

IMPORTANT RULES:
- Both points MUST lie ON the rotation axis -- at the horizontal center of the object at that height, NOT on the lateral silhouette.
- obj_id 1 MUST be above obj_id 2 (smaller Y value).
- The two points MUST be as far apart vertically as possible.
- Both points MUST lie on the visible object surface.
- For flat-topped or flat-bottomed objects, place the pole at the geometric center of the top or bottom face.
- For ROUNDED or CURVED tops/bottoms (no single flat face -- e.g. a dome, a sphere cap, a rounded knob), place the pole at the center of curvature of that rounded region: the point on the visible surface that is equidistant from the left and right silhouette edges at that region's topmost/bottommost extent.
- Verify: draw an imaginary horizontal line through each point -- the point should be equidistant from the left and right edges at that height, regardless of whether the surface there is flat or curved.
- Do NOT place points on the lateral silhouette edges.

Output ONLY:

<points coords="1 1 X1 Y1 2 X2 Y2">"""


PROMPT_MULTI = """You are given multiple views of the SAME 3D object.

The object has ONE dominant axis of rotational symmetry.

For each image, identify the TOP and BOTTOM poles of the global rotation axis -- the points where the axis exits the object's surface at the very top and very bottom.

For each image:
1. Locate where the global axis exits at the top and bottom of the object.
2. Return:
   - obj_id 1: the TOP pole (topmost point on the axis, at the horizontal center of the topmost surface),
   - obj_id 2: the BOTTOM pole (bottommost point on the axis, at the horizontal center of the bottommost surface).

IMPORTANT RULES:
- Both points MUST be ON the rotation axis -- at the horizontal center of the object at that height, NOT on the lateral silhouette.
- Use the SAME global axis consistently across all views.
- obj_id 1 MUST be above obj_id 2 in each image.
- For flat surfaces (e.g., flat-topped objects), place the pole at the geometric center of the top or bottom face.
- For ROUNDED or CURVED tops/bottoms (no single flat face -- e.g. a dome, a sphere cap, a rounded knob), place the pole at the center of curvature of that rounded region: the point equidistant from the left and right silhouette edges at that region's topmost/bottommost extent.
- Infer the global axis from ALL views jointly before answering.
- Verify consistency: the TOP pole should correspond to the same geometric point on the object across all views, whether the surface there is flat or curved.
- Do NOT place points on the lateral silhouette edges.

Output format (one entry per image, separated by semicolons):

<points coords="1 1 Xtop Ytop 2 Xbottom Ybottom; 2 1 Xtop Ytop 2 Xbottom Ybottom; 3 1 Xtop Ytop 2 Xbottom Ybottom">

Where each entry is: image_index obj_id X Y
- obj_id 1 = TOP pole of the rotation axis
- obj_id 2 = BOTTOM pole of the rotation axis

Return ONLY the <points ...> block."""

print(f"Prompt cargado: {PROMPT_ID}")
print(f"(PROMPT_SINGLE: {len(PROMPT_SINGLE)} chars, PROMPT_MULTI: {len(PROMPT_MULTI)} chars)")

## 3. Geometria: camara, triangulacion, filtro fondo/objeto (todo inline)

Copia funcional de `pipeline_common/camera.py` + `pipeline_common/triangulation.py`.

In [ ]:
# --- pipeline_common/camera.py ---

def molmo_to_ndc(x: float, y: float) -> tuple[float, float]:
    """Convert Molmo2 coords (0-1000, top-left origin) to NDC ([-1, 1])."""
    ndc_x = (x / 1000.0) * 2.0 - 1.0
    ndc_y = 1.0 - (y / 1000.0) * 2.0
    return ndc_x, ndc_y


def build_camera_rays(ndc_x: float, ndc_y: float, R: list, T: list,
                      fov_deg: float, image_size: int) -> tuple[np.ndarray, np.ndarray]:
    """World-space (ray_origin, ray_direction) for a given NDC point.
    PyTorch3D row-vector convention: p_cam = p_world @ R + T, so camera center = -(R @ T)."""
    R_np = np.array(R, dtype=np.float64)
    T_np = np.array(T, dtype=np.float64)
    ray_origin = -(R_np @ T_np)
    half_tan = np.tan(np.deg2rad(fov_deg) / 2.0)
    dir_cam = np.array([ndc_x * half_tan, ndc_y * half_tan, 1.0], dtype=np.float64)
    dir_world = R_np @ dir_cam
    dir_world /= np.linalg.norm(dir_world)
    return ray_origin, dir_world


# --- pipeline_common/triangulation.py ---

def ray_dir_for_point(x: float, y: float, R: list, T: list,
                      fov_deg: float, image_size: int) -> tuple[np.ndarray, np.ndarray]:
    ndc_x, ndc_y = molmo_to_ndc(x, y)
    return build_camera_rays(ndc_x, ndc_y, R, T, fov_deg, image_size)


def view_forward_direction(R: list, T: list, fov_deg: float, image_size: int) -> np.ndarray:
    _, direction = build_camera_rays(0.0, 0.0, R, T, fov_deg, image_size)
    return direction


def interpretation_plane_normal(dir_a: np.ndarray, dir_b: np.ndarray):
    """Normal of the plane containing the camera center and two rays from it
    (Bartoli & Sturm 2005). None if the two rays are (numerically) parallel."""
    n = np.cross(dir_a, dir_b)
    norm = np.linalg.norm(n)
    if norm < 1e-9:
        return None
    return n / norm


def triangulate_line(camera_centers: list, plane_normals: list) -> tuple[np.ndarray, np.ndarray]:
    """Intersects >=2 interpretation planes to recover a 3D line: direction =
    right singular vector of smallest singular value; point = least-squares
    solution of n_i . p = n_i . C_i."""
    N = np.asarray(plane_normals, dtype=np.float64)
    C = np.asarray(camera_centers, dtype=np.float64)
    _, _, Vt = np.linalg.svd(N)
    direction = Vt[-1]
    direction /= np.linalg.norm(direction)
    b = np.einsum("ij,ij->i", N, C)
    point, *_ = np.linalg.lstsq(N, b, rcond=None)
    return point, direction


def widest_pair(pts: list):
    """De TODOS los puntos que trae una vista, el par con mayor distancia
    euclidiana en pixeles -- None si hay menos de 2 puntos. Con exactamente 2
    puntos, es identico a tomar esos dos."""
    if len(pts) < 2:
        return None
    best_pair, best_dist_sq = None, -1.0
    for i in range(len(pts)):
        for j in range(i + 1, len(pts)):
            dx = pts[i]["x"] - pts[j]["x"]
            dy = pts[i]["y"] - pts[j]["y"]
            dist_sq = dx * dx + dy * dy
            if dist_sq > best_dist_sq:
                best_dist_sq, best_pair = dist_sq, (pts[i], pts[j])
    return best_pair


def get_point_by_obj_id(pts: list, obj_id: int):
    return next((p for p in pts if p["obj_id"] == obj_id), None)

In [ ]:
# --- filtro fondo/objeto (Mapping/estimate_symmetry_no_mesh.py::filter_points_on_object) ---

BACKGROUND_THRESH = 250  # RGB > esto en los 3 canales = fondo blanco (confirmado: PyTorch3D HardFlatShader default)


def is_on_object(pixel: np.ndarray) -> bool:
    return not bool(np.all(pixel[:3] > BACKGROUND_THRESH))


def molmo_xy_to_pixel(x: float, y: float, img_w: int, img_h: int) -> tuple[int, int]:
    px = int(round((x / 1000.0) * img_w))
    py = int(round((y / 1000.0) * img_h))
    return min(max(px, 0), img_w - 1), min(max(py, 0), img_h - 1)


def filter_points_on_object(points_by_image: dict, images_sent: list, render_dir: Path,
                            min_points: int = 2, image_cache: dict | None = None) -> dict:
    """Descarta puntos que no caen sobre el objeto renderizado. Si a una
    vista le quedan menos de min_points puntos validos, su lista queda vacia
    (= vista invalida para widest_pair/get_point_by_obj_id)."""
    if image_cache is None:
        image_cache = {}
    filtered = {}
    for img_idx_str, pts in points_by_image.items():
        if not pts:
            filtered[img_idx_str] = pts
            continue
        cam = images_sent[int(img_idx_str)]
        filename = cam["filename"]
        if filename not in image_cache:
            img_path = render_dir / filename
            image_cache[filename] = np.array(Image.open(img_path).convert("RGB")) if img_path.exists() else None
        img = image_cache[filename]
        if img is None:
            filtered[img_idx_str] = pts
            continue
        img_h, img_w = img.shape[0], img.shape[1]
        kept = []
        for p in pts:
            px, py = molmo_xy_to_pixel(p["x"], p["y"], img_w, img_h)
            if is_on_object(img[py, px]):
                kept.append(p)
        filtered[img_idx_str] = kept if len(kept) >= min_points else []
    return filtered

## 4. Molmo2: carga del modelo + inferencia (inline) -- necesita GPU

Copia funcional de `MolmoPointing/molmo_multiview_runner.py` (Flow A, sin
Flow B/C -- no hace falta para este sandbox).

In [ ]:
_processor = None
_model     = None


def get_model():
    global _processor, _model
    if _processor is None or _model is None:
        print(f"[model] Loading {MODEL_ID} ...")
        _processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True, device_map="auto", use_fast=True)
        _model = AutoModelForImageTextToText.from_pretrained(
            MODEL_ID, trust_remote_code=True, device_map="auto", dtype=torch.bfloat16,
        )
        _model.eval()
        print("[model] Ready.")
    return _processor, _model


def call_model(images: list, prompt: str) -> str:
    """Un solo llamado con 1..N imagenes. Devuelve el texto crudo decodificado."""
    processor, model = get_model()
    content = [{"type": "text", "text": prompt}]
    for img in images:
        content.append({"type": "image", "image": img})
    messages = [{"role": "user", "content": content}]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.inference_mode():
        output_ids = model.generate(**inputs, max_new_tokens=2048)
    n_input = inputs["input_ids"].size(1)
    del inputs
    text = processor.tokenizer.decode(output_ids[0, n_input:], skip_special_tokens=True)
    del output_ids
    torch.cuda.empty_cache()
    return text


def parse_single_coords(text: str) -> dict:
    """<points coords="RADIO ID X Y ID X Y ..."> -> {"0": [{obj_id,x,y}, ...]}"""
    match = re.search(r'coords=["\']([^"\']+)["\']', text)
    if not match:
        return {}
    raw = [float(n) for n in match.group(1).split()]
    if len(raw) < 4:
        return {}
    raw = raw[1:]
    pts = [{"obj_id": int(raw[i]), "x": raw[i + 1], "y": raw[i + 2]} for i in range(0, len(raw) - 2, 3)]
    return {"0": pts} if pts else {}


def parse_multi_coords(text: str, n_images: int) -> dict:
    """<points coords="img_idx obj_id X Y; ..."> -> {"0": [...], "1": [...], ...} (0-based)"""
    match = re.search(r'coords=["\']([^"\']+)["\']', text)
    if not match:
        return {}
    result: dict = {}
    for group in match.group(1).split(";"):
        group = group.strip()
        if not group:
            continue
        nums = group.split()
        if len(nums) < 4:
            continue
        try:
            img_idx = int(float(nums[0])) - 1
        except ValueError:
            continue
        if img_idx < 0 or img_idx >= n_images:
            continue
        key, rest, pts = str(img_idx), nums[1:], []
        for i in range(0, len(rest) - 2, 3):
            try:
                pts.append({"obj_id": int(float(rest[i])), "x": float(rest[i + 1]), "y": float(rest[i + 2])})
            except ValueError:
                continue
        if pts:
            result.setdefault(key, []).extend(pts)
    return result


def load_metadata(render_dir: Path) -> list:
    path = render_dir / "metadata_all.json"
    if not path.exists():
        raise FileNotFoundError(f"metadata_all.json not found: {render_dir}")
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def get_n_views_entries(metadata: list, n_views: int) -> list:
    """n_views entradas evenly spaced (linspace sobre indices) -- evita
    concentrar camaras cerca de un polo, ver docs/pipeline_sin_malla.md S3.1."""
    total = len(metadata)
    if n_views >= total:
        return sorted(metadata, key=lambda e: e["index"])
    indices = {int(round(i)) for i in np.linspace(0, total - 1, n_views)}
    entries = [m for m in metadata if m["index"] in indices]
    return sorted(entries, key=lambda e: e["index"])


def run_inference(images: list, prompt_single: str, prompt_multi: str) -> tuple[str, dict]:
    """auto mode: 1 imagen -> PROMPT_SINGLE; >1 -> PROMPT_MULTI (1 solo llamado)."""
    if len(images) == 1:
        raw = call_model(images, prompt_single)
        pts = parse_single_coords(raw)
        return raw, ({"0": pts["0"]} if "0" in pts else {})
    raw = call_model(images, prompt_multi)
    return raw, parse_multi_coords(raw, n_images=len(images))

## 5. Correr inferencia sobre los 9 objetos

Guarda `molmo_multiview_<EXPERIMENT_ID>.json` dentro del sandbox (mismo
formato que el pipeline real, generado con el codigo de arriba en vez de
invocar el script). Si el JSON de una vista ya existe, se saltea -- borralo
a mano (o cambia `EXPERIMENT_ID`) para forzar recalculo.

In [ ]:
EXPERIMENT_ID = "axis_v06_sandbox"
VIEW_GROUPS   = [6, 14, 26]

for cat, oid in OBJECT_LIST:
    render_dir = SANDBOX_RENDERS / SYMMETRY_TYPE / oid / str(SIZE) / LIGHTING
    metadata   = load_metadata(render_dir)
    json_path  = render_dir / f"molmo_multiview_{EXPERIMENT_ID}.json"
    results    = json.load(open(json_path, encoding="utf-8")) if json_path.exists() else {}

    for n_views in VIEW_GROUPS:
        if str(n_views) in results:
            continue
        entries = get_n_views_entries(metadata, n_views)
        images  = [Image.open(render_dir / e["filename"]).convert("RGB") for e in entries]

        raw, points_by_img = run_inference(images, PROMPT_SINGLE, PROMPT_MULTI)

        results[str(n_views)] = {
            "experiment_id": EXPERIMENT_ID, "prompt_id": PROMPT_ID,
            "prompt_used": PROMPT_SINGLE if n_views == 1 else PROMPT_MULTI,
            "raw_output": raw, "points_by_image": points_by_img,
            "images_sent": [
                {"img_idx": i, "filename": e["filename"], "index": e["index"],
                 "azimuth": e["azimuth"], "elevation": e["elevation"], "eye": e["eye"],
                 "R": e["R"], "T": e["T"]}
                for i, e in enumerate(entries)
            ],
            "n_points": sum(len(v) for v in points_by_img.values()),
        }
        json_path.parent.mkdir(parents=True, exist_ok=True)
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2)
        print(f"  [{cat:6s}] {oid} n_views={n_views}: {results[str(n_views)]['n_points']} puntos devueltos")

print("\nInferencia completa.")

## 6. Estimar eje sin malla (triangulacion, inline)

Copia funcional de `Mapping/estimate_symmetry_no_mesh.py::estimate_axis_no_mesh`
(pooling de normales de interpretacion entre TODAS las vistas de un
n_views group -> una sola SVD).

In [ ]:
FILTER_OFF_OBJECT    = False   # True para probar el filtro fondo/objeto
MIN_POINTS_ON_OBJECT = 2


def estimate_axis_no_mesh(points_by_image: dict, images_sent: list, fov_deg: float, image_size: int):
    centers, normals = [], []
    for img_idx_str, pts in points_by_image.items():
        pair = widest_pair(pts)
        if pair is None:
            continue
        p_a, p_b = pair
        cam = images_sent[int(img_idx_str)]
        C, d_a = ray_dir_for_point(p_a["x"], p_a["y"], cam["R"], cam["T"], fov_deg, image_size)
        _, d_b = ray_dir_for_point(p_b["x"], p_b["y"], cam["R"], cam["T"], fov_deg, image_size)
        n = interpretation_plane_normal(d_a, d_b)
        if n is None:
            continue
        centers.append(C)
        normals.append(n)
    if len(normals) < 2:
        raise ValueError(f"need >=2 valid views, got {len(normals)}")
    point, direction = triangulate_line(centers, normals)
    return {"direction": direction.tolist(), "origin": point.tolist(), "n_views_used": len(normals)}


predicted_axes = {}   # (object_id, n_views) -> {"direction", "origin"} | None si fallo

for cat, oid in OBJECT_LIST:
    render_dir = SANDBOX_RENDERS / SYMMETRY_TYPE / oid / str(SIZE) / LIGHTING
    manifest   = json.load(open(render_dir / "manifest.json", encoding="utf-8")) if (render_dir / "manifest.json").exists() else {}
    fov_deg    = manifest.get("fov", DEFAULT_FOV)
    image_size = manifest.get("image_size", SIZE)

    json_path = render_dir / f"molmo_multiview_{EXPERIMENT_ID}.json"
    molmo_data = json.load(open(json_path, encoding="utf-8"))

    image_cache = {}
    for n_views_key, group in molmo_data.items():
        points_by_image = group["points_by_image"]
        images_sent     = group["images_sent"]
        if FILTER_OFF_OBJECT:
            points_by_image = filter_points_on_object(
                points_by_image, images_sent, render_dir,
                min_points=MIN_POINTS_ON_OBJECT, image_cache=image_cache,
            )
        try:
            pred = estimate_axis_no_mesh(points_by_image, images_sent, fov_deg, image_size)
        except ValueError as e:
            pred = None
            print(f"  [{cat:6s}] {oid} n_views={n_views_key}: [omitido] {e}")
        predicted_axes[(oid, int(n_views_key))] = pred

print(f"\n{sum(v is not None for v in predicted_axes.values())}/{len(predicted_axes)} predicciones validas.")

## 7. Evaluar contra GT (inline)

Copia funcional de `Mapping/evaluate.py::parse_true_label` /
`angular_error_deg` / `point_to_line_distance` / `precision_{t}deg`.

In [ ]:
ANGULAR_THRESHOLDS = [5, 10, 15]


def parse_true_label(txt_path: Path) -> dict:
    lines = [l.strip() for l in txt_path.read_text().splitlines() if l.strip()]
    for line in lines:
        if line.startswith("axis"):
            parts = line.split()
            vec  = np.array([float(x) for x in parts[1:4]])
            orig = [float(x) for x in parts[4:7]]
            vec /= np.linalg.norm(vec)
            return {"direction": vec.tolist(), "origin": orig}
    raise ValueError(f"No se encontro una linea 'axis' en {txt_path}")


def angular_error_deg(v1: np.ndarray, v2: np.ndarray) -> float:
    v1, v2 = v1 / np.linalg.norm(v1), v2 / np.linalg.norm(v2)
    return float(np.degrees(np.arccos(np.clip(np.abs(np.dot(v1, v2)), 0.0, 1.0))))


def point_to_line_distance(point: np.ndarray, line_origin: np.ndarray, line_dir: np.ndarray) -> float:
    d = line_dir / np.linalg.norm(line_dir)
    v = point - line_origin
    return float(np.linalg.norm(v - np.dot(v, d) * d))


rows = []
for cat, oid in OBJECT_LIST:
    gt = parse_true_label(SANDBOX_OBJECTS_DIR / f"{oid}.txt")
    t_dir, t_orig = np.array(gt["direction"]), np.array(gt["origin"])

    for n_views in VIEW_GROUPS:
        pred = predicted_axes.get((oid, n_views))
        row = {"categoria": cat, "object_id": oid, "n_views": n_views}
        if pred is None:
            row.update({"status": "no_pred", "angular_error_deg": 90.0, "translation_error": None})
        else:
            p_dir, p_orig = np.array(pred["direction"]), np.array(pred["origin"])
            ang  = angular_error_deg(p_dir, t_dir)
            dist = point_to_line_distance(p_orig, t_orig, t_dir)
            row.update({"status": "ok", "angular_error_deg": round(ang, 4), "translation_error": round(dist, 6)})
            for t in ANGULAR_THRESHOLDS:
                row[f"precision_{t}deg"] = int(ang < t)
        rows.append(row)

df_eval = pd.DataFrame(rows).sort_values(["categoria", "object_id", "n_views"]).reset_index(drop=True)
df_eval

## 8. Resultados: resumen por categoria y por n_views

In [ ]:
print(f"Experimento: {EXPERIMENT_ID}  |  Prompt: {PROMPT_ID}  |  "
      f"filter_off_object={FILTER_OFF_OBJECT}\n")

print("--- Por objeto ---")
display(df_eval)

print("\n--- Promedio por categoria x n_views ---")
display(df_eval.groupby(["categoria", "n_views"])[["angular_error_deg"]].mean().round(2))

print("\n--- Promedio global por n_views (equivalente a angular_error_mean del pipeline real) ---")
display(df_eval.groupby("n_views")[["angular_error_deg"]].agg(["mean", "median", "std", "min", "max"]).round(2))

## 9. Visualizacion 2D/3D (opcional)

Mismo esquema de `Experiments/visualizar_casos_axis.ipynb`, reescrito inline
(sin importar nada del repo salvo `trimesh`/`PIL`/`matplotlib`, ya cargados
en el Setup).

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401


def plot_case(object_id: str, categoria: str, n_views: int, max_views_2d: int = 6) -> None:
    render_dir = SANDBOX_RENDERS / SYMMETRY_TYPE / object_id / str(SIZE) / LIGHTING
    molmo_data = json.load(open(render_dir / f"molmo_multiview_{EXPERIMENT_ID}.json", encoding="utf-8"))
    group = molmo_data[str(n_views)]
    images_sent, points_by_image = group["images_sent"], group["points_by_image"]

    pred = predicted_axes.get((object_id, n_views))
    gt   = parse_true_label(SANDBOX_OBJECTS_DIR / f"{object_id}.txt")
    err  = df_eval[(df_eval.object_id == object_id) & (df_eval.n_views == n_views)]["angular_error_deg"].iloc[0]

    # --- 2D ---
    idxs = sorted(points_by_image.keys(), key=int)[:max_views_2d]
    fig, axes = plt.subplots(1, len(idxs), figsize=(3.2 * len(idxs), 3.4))
    if len(idxs) == 1:
        axes = [axes]
    for ax, idx_str in zip(axes, idxs):
        cam = images_sent[int(idx_str)]
        img_path = render_dir / cam["filename"]
        img = np.array(Image.open(img_path)) if img_path.exists() else None
        if img is not None:
            ax.imshow(img)
            img_h, img_w = img.shape[0], img.shape[1]
        else:
            img_w = img_h = SIZE
            ax.set_xlim(0, img_w); ax.set_ylim(img_h, 0)
        for p in points_by_image[idx_str]:
            px, py = molmo_xy_to_pixel(p["x"], p["y"], img_w, img_h)
            ax.scatter([px], [py], c="red" if p["obj_id"] == 1 else "blue",
                       s=60, edgecolors="white", linewidths=1.2, zorder=5)
        ax.set_title(f"img {idx_str}", fontsize=8)
        ax.axis("off")
    fig.suptitle(f"[{categoria}] {object_id} -- angular_error={err:.2f} grados", fontsize=10)
    fig.tight_layout()
    plt.show()

    # --- 3D ---
    mesh  = trimesh.load(str(SANDBOX_OBJECTS_DIR / f"{object_id}.obj"), force="mesh", process=False)
    verts = np.asarray(mesh.vertices)
    if len(verts) > 4000:
        verts_plot = verts[np.random.default_rng(0).choice(len(verts), 4000, replace=False)]
    else:
        verts_plot = verts
    bbox_diag = float(np.linalg.norm(verts.max(axis=0) - verts.min(axis=0)))

    fig = plt.figure(figsize=(6.5, 6.5))
    ax = fig.add_subplot(111, projection="3d")
    ax.scatter(verts_plot[:, 0], verts_plot[:, 1], verts_plot[:, 2], s=1, c="lightgray", alpha=0.4, label="malla")

    def plot_line(origin, direction, color, label):
        d = np.array(direction) / np.linalg.norm(direction)
        o = np.array(origin)
        p0, p1 = o - d * bbox_diag * 0.75, o + d * bbox_diag * 0.75
        ax.plot([p0[0], p1[0]], [p0[1], p1[1]], [p0[2], p1[2]], color=color, linewidth=2.5, label=label)

    plot_line(gt["origin"], gt["direction"], "green", "eje GT")
    if pred is not None:
        plot_line(pred["origin"], pred["direction"], "red", "eje predicho")
    ax.set_title(f"[{categoria}] {object_id} -- angular_error={err:.2f} grados", fontsize=10)
    ax.legend(loc="upper left", fontsize=8)
    ax.set_box_aspect([1, 1, 1])
    fig.tight_layout()
    plt.show()


# Ejemplo: un caso de cada categoria, al mayor n_views
for cat, oid in [(c, o) for c, o in OBJECT_LIST if c in ("BUENO", "MEDIO", "MALO")][::3]:
    plot_case(oid, cat, n_views=max(VIEW_GROUPS))